# PVE Signal Sensitivity Analysis

Does increasing the proportion of variance explained (PVE) in the data
make our agents' responses more distinguishable from the null?

For each PVE level (0.0, 0.01, 0.1) and each dataset, we pool the PVE
responses as the alt group and the existing null responses as the null
group, then run our stability checks.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")

BASE_DIR = Path("..").resolve()
sys.path.insert(0, str(BASE_DIR / "scripts"))

from checks import bootstrap_mean_test, overlap_coefficient

In [ ]:
# Load data
pve_path = BASE_DIR / "aggregated_results" / "aggregated_pve_results.csv"
null_path = BASE_DIR / "aggregated_results" / "aggregated_results.csv"

if not pve_path.exists():
    raise FileNotFoundError(
        f"Missing {pve_path.name}. Run scripts/aggregate_pve_conclusions.py first."
    )
if not null_path.exists():
    raise FileNotFoundError(
        f"Missing {null_path.name}. Run scripts/aggregate_conclusions.py first."
    )

df_pve = pd.read_csv(pve_path, keep_default_na=False)
df_null = pd.read_csv(null_path, keep_default_na=False)

# keep only null-distribution rows from the main results
df_null = df_null[df_null["distribution"] == "null"].copy()
# only keep rows corresponding to pve levels in [0, 0.01, 0.1]
df_pve = df_pve[df_pve["pve_level"].isin([0, 0.01, 0.1])].copy()
# for pve, only keep rows corresponding to runs 1-5
df_pve = df_pve[df_pve["run_id"].between(1, 5)].copy()
df_null = df_null[df_null["run_id"].between(1, 20)].copy()
# for both null and pve, only keep rows corresponding to perturbations:
# ["anonymize" "shuffle_names" "add_features" "positive_leading_statement" "negative_leading_statement"]
perturbations = [
    "anonymize",
    "shuffle_names",
    "add_features",
    "positive_leading_statement",
    "negative_leading_statement",
]
df_pve = df_pve[df_pve["perturbation"].isin(perturbations)].copy()
df_null = df_null[df_null["perturbation"].isin(perturbations)].copy()

pve_levels = sorted(df_pve["pve_level"].unique(), key=float)
datasets = sorted(df_pve["dataset"].unique())

print(f"PVE levels: {pve_levels}")
print(f"Datasets: {datasets}")
print(f"PVE rows: {len(df_pve)}, null rows: {len(df_null)}")

In [ ]:
# get the number of rows for each dataset in pve
for dataset in datasets:
    n_pve = len(df_pve[df_pve["dataset"] == dataset])
    n_null = len(df_null[df_null["dataset"] == dataset])
    print(f"{dataset}: {n_pve} pve rows, {n_null} null rows")

In [ ]:
# Bootstrap mean test + overlap coefficient per (pve_level, dataset)
rng = np.random.default_rng(42)
OVL_THRESHOLD = 0.2
MEAN_ALPHA = 0.05

results = []
for level in pve_levels:
    for ds in datasets:
        pve_scores = df_pve.loc[
            (df_pve["dataset"] == ds) & (df_pve["pve_level"] == level),
            "response",
        ].values.astype(float)
        null_scores = df_null.loc[df_null["dataset"] == ds, "response"].values.astype(float)

        if len(pve_scores) == 0 or len(null_scores) == 0:
            continue

        obs_mean, p_value, _, ci_95 = bootstrap_mean_test(pve_scores, mu0=50.0, rng=rng)
        ovl = overlap_coefficient(pve_scores, null_scores)

        results.append({
            "dataset": ds,
            "pve_level": float(level),
            "n_pve": len(pve_scores),
            "n_null": len(null_scores),
            "obs_mean": obs_mean,
            "p_value": p_value,
            "ci_lo": ci_95[0],
            "ci_hi": ci_95[1],
            "ovl": ovl,
        })

results_df = pd.DataFrame(results)
results_df["mean_sig"] = results_df["p_value"] < MEAN_ALPHA
results_df["low_ovl"] = results_df["ovl"] < OVL_THRESHOLD

print(results_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# Scatterplot: bootstrap mean vs overlap coefficient — panelled row, one panel per PVE level
from adjustText import adjust_text

categories = ["Confident Yes", "Low Signal", "Analysis Failure", "No Signal"]
cat_colors = {
    "Confident Yes":    "seagreen",
    "Low Signal":       "goldenrod",
    "Analysis Failure": "mediumpurple",
    "No Signal":        "lightcoral",
}

LABEL_FS = 24
TICK_FS  = 20
ANNOT_FS = 18
TITLE_FS = 28
LEGEND_FS = 20

def classify(row):
    if row["mean_sig"] and row["low_ovl"]:
        return "Confident Yes"
    elif row["mean_sig"] and not row["low_ovl"]:
        return "Analysis Failure"
    elif not row["mean_sig"] and row["low_ovl"]:
        return "Low Signal"
    else:
        return "No Signal"

results_df["category"] = results_df.apply(classify, axis=1)

fig, axes = plt.subplots(1, len(pve_levels), figsize=(7 * len(pve_levels), 6), sharey=True)

for ax, level in zip(axes, pve_levels):
    sub_level = results_df[results_df["pve_level"] == level]

    texts = []
    for cat in categories:
        sub = sub_level[sub_level["category"] == cat]
        ax.scatter(
            sub["ovl"], sub["obs_mean"],
            color=cat_colors[cat], s=200, label=cat,
            edgecolor="white", linewidth=0.6, zorder=3,
        )
        for _, row in sub.iterrows():
            texts.append(ax.text(
                row["ovl"], row["obs_mean"], row["dataset"],
                fontsize=ANNOT_FS, fontfamily="DejaVu Sans Mono",
            ))
            
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-5, 105)

    ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, zorder=1)
    ax.axvline(OVL_THRESHOLD, color="gray", linestyle="--", linewidth=0.8, zorder=1)

    ax.set_xlabel("Overlap Coefficient", fontsize=LABEL_FS)
    if ax is axes[0]:
        ax.set_ylabel("PVE Response Mean", fontsize=LABEL_FS)
    ax.tick_params(labelsize=TICK_FS)
    ax.set_title(f"PVE = {float(level):.2f}", fontsize=TITLE_FS, fontweight="bold")
    if ax is axes[0]:
        ax.legend(loc="upper left", fontsize=LEGEND_FS)
        
    adjust_text(
        texts, ax=ax,
        expand=(1.3, 1.5),
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.6),
    )

sns.despine()
plt.tight_layout()
plt.savefig(BASE_DIR / "insights" / "figures" / "pve_signal_analysis.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# Ridgeline (ggridges-style) density plots — panelled row, one panel per PVE level
from scipy.stats import gaussian_kde
from matplotlib.patches import Patch

X_GRID      = np.linspace(0, 100, 500)
ROW_HEIGHT  = 1.0
RIDGE_SCALE = 0.75
NULL_COLOR  = "#008080"
PVE_COLOR   = "#FF7F50"

LABEL_FS = 24
TICK_FS  = 20
TITLE_FS = 28
LEGEND_FS = 20

n_ds = len(datasets)
fig, axes = plt.subplots(1, len(pve_levels), figsize=(7 * len(pve_levels), n_ds * 0.7 + 1.5), sharey=True)

for ax, level in zip(axes, pve_levels):
    for i, ds in enumerate(reversed(datasets)):
        baseline    = i * ROW_HEIGHT
        pve_scores  = df_pve.loc[
            (df_pve["dataset"] == ds) & (df_pve["pve_level"] == level), "response"
        ].values.astype(float)
        null_scores = df_null.loc[df_null["dataset"] == ds, "response"].values.astype(float)

        ax.axhline(baseline, color="gray", linewidth=0.4, zorder=0)

        for scores, color in [(null_scores, NULL_COLOR), (pve_scores, PVE_COLOR)]:
            if len(scores) < 2:
                continue
            kde  = gaussian_kde(scores)
            dens = kde(X_GRID)
            dens = dens / dens.max() * RIDGE_SCALE
            ax.fill_between(X_GRID, baseline, baseline + dens,
                            color=color, alpha=0.45, linewidth=0)
            ax.plot(X_GRID, baseline + dens, color=color, linewidth=1.2)

        if ax is axes[0]:
            ax.text(-1, baseline + RIDGE_SCALE * 0.1, ds,
                    ha="right", va="bottom", fontsize=TICK_FS,
                    fontfamily="DejaVu Sans Mono")

    ax.set_xlim(0, 100)
    ax.set_ylim(-ROW_HEIGHT * 0.3, n_ds * ROW_HEIGHT + 0.2)
    ax.set_xlabel("Response Score", fontsize=LABEL_FS)
    ax.set_yticks([])
    ax.tick_params(axis="x", labelsize=TICK_FS)
    ax.set_title(f"PVE = {float(level):.2f}", fontsize=TITLE_FS, fontweight="bold")

axes[-1].legend(
    handles=[Patch(color=NULL_COLOR, alpha=0.7, label="Null"),
             Patch(color=PVE_COLOR,  alpha=0.7, label="PVE")],
    loc="upper right", fontsize=LEGEND_FS,
)
sns.despine(left=True)
plt.tight_layout()
plt.savefig(BASE_DIR / "insights" / "figures" / "pve_signal_analysis_ridges.png", dpi=600, bbox_inches="tight")
plt.show()